# Chunking strategies
Chunking helps in splitting a large file into smaller chunks or nodes so that when a user makes a query we just get the chunks that are necessary and then pass them to the LLm together with the query.
hence choosing the right chunking method for the given data is crucial

In [1]:
# we first initialise the env vars
from dotenv import load_dotenv

load_dotenv("../.env")

True

In [2]:
# we read the data from the book in the data folder
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(input_files=["./data/The-Psychology-of-Money-Morgan-Housel.pdf"]).load_data()

## Fixed size chunking

When it comes to fixed size chunking the problem could be spliting a sentence midway hence the chunk will be incomplete, the overlap tries to minimize the damage

In [3]:
# we now create the tokenSplitter
from llama_index.core.node_parser import TokenTextSplitter

splitter = TokenTextSplitter(
    chunk_size=512,
    chunk_overlap=50
)

fixedSizeChunkingNodes = splitter.get_nodes_from_documents(documents)

In [4]:
print(
    f"""Node type: {type(fixedSizeChunkingNodes)}
Number Nodes: {len(fixedSizeChunkingNodes)}\n\n
First Node;\n{fixedSizeChunkingNodes[0]}\n\n
Second Node;\n{fixedSizeChunkingNodes[1]}\n\n
Third Node;\n{fixedSizeChunkingNodes[2]}\n\n
200th Node;\n{fixedSizeChunkingNodes[199]}"""
)

Node type: <class 'list'>
Number Nodes: 226


First Node;
Node ID: 994cedbf-b97b-4454-98a0-b13e3737eafe
Text:


Second Node;
Node ID: 36827cb4-6d26-4ecc-965f-83eaa2493534
Text: For My parents, who teach me. Gretchen, who guides me. Miles and
Reese, who inspire me.


Third Node;
Node ID: ea913d95-87ad-4f04-9fb5-d55df9fe3c51
Text: Introduction: The Greatest Show On Earth   1. No One’s Crazy
2. Luck & Risk   3. Never Enough


200th Node;
Node ID: f92874ae-afe0-446a-af81-8a0276b70420
Text: The enormous lead of the well-to-do in the economic race has
been considerably reduced. It is the industrial workers who as a group
have done best—people such as a steelworker’s family who used to live
on $2,500 and now are getting $4,500, or the highly skilled machine-
tool operator’s family who used to have $3,000 and now can spend an
annual $5...


## Sentence Based Chunking
This is where the chunks are complete sentences up to the chunk size limit. good general chunking method

In [5]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(
    chunk_size=512,
    chunk_overlap=50,
)

sentenceBasedChunkingNodes = splitter.get_nodes_from_documents(documents)

In [6]:
# we take a quick look at the nodes
print(f"""
Nodes Type: {type(sentenceBasedChunkingNodes)}
Number Nodes: {len(sentenceBasedChunkingNodes)}

First Node;
{sentenceBasedChunkingNodes[0]}

Second Node;
{sentenceBasedChunkingNodes[1]}

Third Node;
{sentenceBasedChunkingNodes[2]}

200th Node;
{sentenceBasedChunkingNodes[199]}
""")


Nodes Type: <class 'list'>
Number Nodes: 225

First Node;
Node ID: 4d55ee37-5926-4a81-ad2b-1d03e5f4e8d0
Text:

Second Node;
Node ID: cf4f4a93-0c08-4da6-8703-1bcb2a99d84c
Text: For My parents, who teach me. Gretchen, who guides me. Miles and
Reese, who inspire me.

Third Node;
Node ID: 5395711b-7692-4977-8aea-e48a3f5f425d
Text: Introduction: The Greatest Show On Earth   1. No One’s Crazy
2. Luck & Risk   3. Never Enough

200th Node;
Node ID: 3cd9f2a7-b36d-423e-a8d6-37317a71d66c
Text: The leveling out of classes meant a leveling out of lifestyles.
Normal people drove Chevys. Rich people drove Cadillacs. TV and radio
equalized the entertainment and culture people enjoyed regardless of
class. Mail-order catalogs equalized the clothes people wore and the
goods they bought regardless of where they lived. Harper’s Magazine
noted i...



## Sentence Window Chunking
The other type of splitter that is similar to this one is the sentenceWindow splitter that also adds the sentences above and below the current node sentence, this is good as it allows for getting the meaning from the surrounding sentences as well.
the downside is if the surrounding sentences are not closely related to the current node main sentence

In [7]:
from llama_index.core.node_parser import SentenceWindowNodeParser

splitter = SentenceWindowNodeParser(
    window_size=7
)

sentenceWindowNodes = splitter.get_nodes_from_documents(documents)

In [8]:
# let us take a look at the node that have been generated

print(f"""
Number Nodes: {len(sentenceWindowNodes)}

1st Node;
{sentenceWindowNodes[0]}

2nd Node;
{sentenceWindowNodes[1]}

3rd Node;
{sentenceWindowNodes[2]}

200th Node;
{sentenceWindowNodes[199]}
""")


Number Nodes: 3443

1st Node;
Node ID: 4ef3c0c4-6dff-4826-aa14-8fe0e4cbb61a
Text: For My parents, who teach me.

2nd Node;
Node ID: d3a008f4-b291-4f6a-a16f-9a54863eea75
Text: Gretchen, who guides me.

3rd Node;
Node ID: 9f4b5ca4-27cd-4f6a-a7bb-82222042c372
Text: Miles and Reese, who inspire me.

200th Node;
Node ID: d6a75769-2a23-4799-b37b-9dc78cfaced0
Text: But they can’t model the feeling of coming home, looking at your
kids, and wondering if you’ve made a mistake that will impact their
lives.



As noted from the output the number of nodes that have been generated by this one are more than the ones that were generated by the other two, the fixed size method and the sentence splitter.

## Semantic Chunking
This is where the chunking is based on the semantic meaning. when the topic shifts significantly (above a certain threashold) the chunk is created and made.
This ensures that the chunks created are semantically similar

In [10]:
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.embeddings.openai import OpenAIEmbedding

# seeing that this requires the embedding of each sentence in the doc, we will use a shorter file
essayDocuments = SimpleDirectoryReader(input_files=["./data/paul_graham_essay.txt"]).load_data()

embed_model = OpenAIEmbedding(model="text-embedding-ada-002")

splitter = SemanticSplitterNodeParser(
    embed_model=embed_model,
    buffer_size=2,
    # i have left the breakpoint percentile as the default 95
    breakpoint_percentile_threshold=95
)

semanticSplitterNodes = splitter.get_nodes_from_documents(essayDocuments)

2026-03-25 11:40:49,594 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-25 11:40:51,420 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-25 11:40:53,281 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-25 11:40:54,691 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-25 11:40:55,997 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-25 11:40:57,289 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-25 11:40:59,175 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-25 11:41:00,388 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [12]:
# let us take a look at the node that have been generated

print(f"""
Number Nodes: {len(semanticSplitterNodes)}

1st Node;
{semanticSplitterNodes[0]}

2nd Node;
{semanticSplitterNodes[1]}

3rd Node;
{semanticSplitterNodes[2]}

10th Node;
{semanticSplitterNodes[10]}
""")



Number Nodes: 39

1st Node;
Node ID: 2f1a2ada-9c3e-48ac-a1bd-0611aa125851
Text: What I Worked On  February 2021  Before college the two main
things I worked on, outside of school, were writing and programming. I
didn't write essays. I wrote what beginning writers were supposed to
write then, and probably still are: short stories.

2nd Node;
Node ID: 91c3ae2e-fb42-4cdf-93e7-6cd5f23d7c65
Text: My stories were awful. They had hardly any plot, just characters
with strong feelings, which I imagined made them deep.  The first
programs I tried writing were on the IBM 1401 that our school district
used for what was then called "data processing." This was in 9th
grade, so I was 13 or 14. The school district's 1401 happened to be in
the basem...

3rd Node;
Node ID: ce63f06e-9c4f-4d22-8f7f-44a1b80c94ef
Text: It seemed, to my naive high school self, to be the study of the
ultimate truths, compared to which the things studied in other fields
would be mere domain knowledge. What I discovered when I

As observed from the output the nodes/chunks have the same semantic meaning in them.
- The first node for example shows the things that the write did.